# 3. Baselines, Anomalies & the Hunting Maturity Model

The previous notebooks used **signatures** (known-bad IPs, known attack tools, fixed thresholds). Real attackers know your signatures — so mature hunting also looks for **anomalies**: activity that deviates from *this environment's normal baseline*.

This notebook teaches:

1. How to build a **baseline** from historical data.
2. How to detect **anomalies** (off-hours sign-ins, unusual volumes).
3. The **hunting maturity model** — from ad-hoc to automated.
4. A small **MITRE ATT&CK coverage** view over the hunts you've written.

We'll inject a small time-spread dataset (7 days) just for this notebook so the baseline math is meaningful. This does not affect the other labs.


In [ ]:
import httpx, json, random, statistics
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone

SIEM = 'http://localhost:8000'

def _iso(dt):
    return dt.replace(tzinfo=None).isoformat()

USER = 'bob@contoso.com'
random.seed(42)
now = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)

# ---- Step 0: learn what "normal" already looks like for bob in this SIEM ----------
# A baseline you INVENT teaches nothing: if it shares no IPs, locations or apps with the
# data already in the SIEM, then every real event looks novel and your anomaly detector
# fires on everything. So we derive bob's normal profile from his existing rows.
existing = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs', 'filter': {'UserPrincipalName': USER}, 'limit': 1000,
}).json()['results']

NORMAL_PROFILE = sorted({
    (e['IPAddress'], e['Location'], e.get('AppDisplayName') or 'Outlook')
    for e in existing
    if e['IPAddress'].startswith('10.')          # corporate ranges only
}) or [('10.0.1.11', 'Seattle', 'Outlook')]      # fallback for an empty SIEM

# bob's working window: a 9-hour block that INCLUDES the hour you are running this.
# (Pinning it to 09:00-17:00 UTC would make the whole demo depend on your local clock:
#  run the lab at 03:00 and every routine sign-in is "off-hours".)
WORK_HOURS = [(now.hour - 8 + i) % 24 for i in range(9)]

print(f"bob's normal profile learned from {len(existing)} existing rows:")
for ip, loc, app in NORMAL_PROFILE:
    print(f'   {ip:<14} {loc:<10} {app}')
print(f'   working hours (UTC): {WORK_HOURS[0]:02d}:00-{WORK_HOURS[-1]:02d}:00\n')

# ---- Step 1: 7 days of NORMAL history, drawn from that profile ------------------
entries = []
for days_ago in range(7, 0, -1):
    day = now - timedelta(days=days_ago)
    for _ in range(5):
        ip, loc, app = random.choice(NORMAL_PROFILE)
        ts = day.replace(hour=random.choice(WORK_HOURS), minute=random.randint(0, 59), second=0)
        entries.append({
            'table_name': 'SigninLogs',
            'timestamp': _iso(ts),
            'data': {'UserPrincipalName': USER, 'IPAddress': ip, 'Location': loc,
                     'ResultType': 'Success', 'AppDisplayName': app},
        })

# ---- Step 2: the anomaly. Same account, brand-new IP / country / app, right now ----
# Note the attacker signs in during bob's WORKING HOURS -- a competent one would.
# That deliberately neutralises the "off-hours" signal, which is the whole lesson.
for _ in range(3):
    ts = now - timedelta(minutes=random.randint(1, 55))
    entries.append({
        'table_name': 'SigninLogs',
        'timestamp': _iso(ts),
        'data': {'UserPrincipalName': USER, 'IPAddress': '185.220.101.42',
                 'Location': 'Moscow', 'ResultType': 'Success',
                 'AppDisplayName': 'Azure Portal'},
    })

r = httpx.post(f'{SIEM}/ingest/batch', json={'entries': entries})
print(f'Injected {len(entries)} time-spread sign-ins for baseline analysis: {r.json()}')


## Step 1 — Build a baseline (hour-of-day profile)

We want to answer: *what does a normal day look like for this user?*  In real KQL, this is one query:

```kusto
SigninLogs
| where UserPrincipalName == "bob@contoso.com"
| where TimeGenerated between (ago(7d) .. ago(1h))
| summarize Signins = count() by Hour = hourofday(TimeGenerated)
| sort by Hour asc
```

> ⚠️ **Do not write `bin(TimeGenerated % 1d, 1h)`.** KQL has no modulo operator for
> `datetime`, so that expression is a syntax error. To collapse a timestamp to an
> hour-of-day bucket use **`hourofday(TimeGenerated)`** (returns 0–23), or
> `bin(TimeGenerated - startofday(TimeGenerated), 1h)` if you want a `timespan`.
> `bin(TimeGenerated, 1h)` is different again — that keeps the date and buckets the
> *timeline* into hours, which is what you want for `make-series`, not for a
> hour-of-day profile.

Our SIEM API doesn't compute `bin()` server-side, so we group in Python — but the *concept* is identical.


In [ ]:
def query(table, filter=None, limit=1000):
    return httpx.post(f'{SIEM}/query', json={
        'table_name': table, 'filter': filter, 'limit': limit
    }).json()['results']

signins = query('SigninLogs', filter={'UserPrincipalName': 'bob@contoso.com'})
print(f'Total sign-ins for bob: {len(signins)}\n')

# Split: baseline (older than 1h) vs recent (last 1h)
cutoff = datetime.now(timezone.utc).replace(tzinfo=None) - timedelta(hours=1)
def _parse(ts):
    return datetime.fromisoformat(ts.replace('Z',''))

baseline = [s for s in signins if _parse(s['timestamp']) < cutoff]
recent   = [s for s in signins if _parse(s['timestamp']) >= cutoff]

baseline_hours = Counter(_parse(s['timestamp']).hour for s in baseline)
print('Hour-of-day baseline (last 7d, excluding the current hour):')
for h in range(24):
    bar = '█' * baseline_hours.get(h, 0)
    marker = "  ← bob's working hours" if h in WORK_HOURS else ''
    print(f'  {h:02d}:00  {baseline_hours.get(h,0):>2} {bar}{marker}')
print('\nThis profile IS the baseline. Everything below is measured against it.')


## Step 2 — Detect anomalies against the baseline

Two simple, widely-used techniques:

| Technique | What it catches |
|-----------|-----------------|
| **Off-hours activity** | An event at an hour where the baseline is ~0. Cheap and effective. |
| **Statistical outlier (mean + N·σ)** | A daily/hourly count that is several standard deviations above the mean. Used by UEBA tools. |

Real KQL examples:

```kusto
// 1. Off-hours
SigninLogs
| extend Hour = hourofday(TimeGenerated)
| where Hour !between (9 .. 17)

// 2. Statistical outlier using series_decompose_anomalies.
//    NOTE: the function returns THREE columns (flags, scores, baseline), so you MUST
//    destructure them. `| extend anomalies = series_decompose_anomalies(Count)` is wrong.
SigninLogs
| where TimeGenerated > ago(14d)
| make-series Count = count()
      on TimeGenerated from ago(14d) to now() step 1h
      by UserPrincipalName
| extend (AnomalyFlags, AnomalyScores, Baseline) =
      series_decompose_anomalies(Count, 2.5, -1, 'linefit')
//        arg 2 = threshold in sigmas, arg 3 = seasonality (-1 = autodetect), arg 4 = trend
| mv-expand TimeGenerated to typeof(datetime),
            Count to typeof(long),
            AnomalyFlags to typeof(int),
            AnomalyScores to typeof(double),
            Baseline to typeof(double)
| where AnomalyFlags == 1        //  1 = spike, -1 = dip, 0 = normal
| project TimeGenerated, UserPrincipalName, Count, Baseline, AnomalyScores
```

**Things the exam checks about `make-series` + `series_decompose_anomalies`:**

- `make-series` produces **one row per group whose columns are arrays** — you cannot
  `where` on the values until you `mv-expand` them back into rows.
- Always give `make-series` a `from … to … step …` and `default=0`; otherwise gaps in
  the data become missing buckets instead of zeros, and the model learns the wrong shape.
- `series_decompose_anomalies` returns `(flags, scores, baseline)` in that order.
  Flags are `1` (spike above baseline), `-1` (dip below), `0` (normal).
- The sibling functions are `series_decompose()` (baseline/seasonal/trend/residual) and
  `series_outliers()` (Tukey-style scores, no seasonality model).


In [ ]:
# --- Anomaly 1: score each recent sign-in against SEVERAL baseline dimensions ---
#
# A single off-hours check is too blunt: run this notebook at 03:00 and every routine
# sign-in looks anomalous. That is precisely the false-positive machine juniors build.
# Real UEBA scores an event on MULTIPLE weak signals and adds them up:
#     unusual hour  +  never-seen IP  +  never-seen location  +  new application
# One weak signal is noise. Three together is an investigation.
print('=== Anomaly scoring: every sign-in in the last hour, vs the 7-day baseline ===\n')

BUSINESS = set(WORK_HOURS)   # learned above, not hard-coded to 09:00-17:00

baseline_ips   = {s['IPAddress'] for s in baseline}
baseline_locs  = {s['Location'] for s in baseline}
baseline_apps  = {s.get('AppDisplayName') for s in baseline}


def score_signin(s):
    """Return (score, [reasons]) for one recent sign-in."""
    hour = _parse(s['timestamp']).hour
    reasons, score = [], 0
    if baseline_hours.get(hour, 0) == 0:
        score += 1
        reasons.append(f'hour {hour:02d}:00 never seen in baseline')
    elif hour not in BUSINESS:
        reasons.append(f'hour {hour:02d}:00 outside business hours (but seen before)')
    if s['IPAddress'] not in baseline_ips:
        score += 2
        reasons.append(f'IP {s["IPAddress"]} never seen for this user')
    if s['Location'] not in baseline_locs:
        score += 2
        reasons.append(f'location {s["Location"]} never seen for this user')
    if s.get('AppDisplayName') and s.get('AppDisplayName') not in baseline_apps:
        score += 1
        reasons.append(f'app {s["AppDisplayName"]} never used by this user')
    return score, reasons


scored = sorted(((score_signin(s), s) for s in recent), key=lambda x: x[0][0], reverse=True)

if not scored:
    print('  No sign-ins in the last hour.')
for (score, reasons), s in scored:
    icon = '🔴' if score >= 4 else ('🟡' if score >= 2 else '⬜')
    verdict = 'INVESTIGATE' if score >= 4 else ('watch' if score >= 2 else 'consistent with baseline')
    print(f'  {icon} score={score}  {s["timestamp"][:19]}  {s["UserPrincipalName"]:<18} '
          f'{s["IPAddress"]:<16} {s["Location"]:<10} → {verdict}')
    for r in reasons:
        print(f'         · {r}')

print()
print('Read the reasons, not just the scores:')
print()
print('  • The attacker signed in DURING bob\'s working hours, so the "unusual hour"')
print('    signal contributed NOTHING. A time-of-day rule on its own would have missed')
print('    this entirely -- which is what a competent attacker is counting on.')
print('  • The routine corporate sign-ins score ~0: familiar IP, familiar location,')
print('    familiar app. A naive "off-hours" rule would have paged you for every one of')
print('    them at 3 a.m. and taught the SOC to ignore the alert.')
print('  • Only the Moscow sign-ins cross the threshold, and they do it on the strength')
print('    of THREE independent weak signals (new IP + new country + new app), none of')
print('    which would justify an alert alone.')
print()
print('That accumulation is exactly what Sentinel UEBA\'s investigation priority score does,')
print('and why it catches things no static threshold rule can.')


In [ ]:
# --- Anomaly 2: statistical outlier on per-day counts ---
print('\n=== Anomaly check: daily volume outlier (mean ± 2σ) ===\n')

per_day = Counter(_parse(s['timestamp']).date() for s in signins)
# Drop today (partial day) from the baseline sample
today = datetime.now(timezone.utc).date()
sample = [c for d, c in per_day.items() if d != today]
today_count = per_day.get(today, 0)

if len(sample) >= 2:
    mean = statistics.mean(sample)
    stdev = statistics.pstdev(sample) or 1.0  # guard against 0
    upper = mean + 2 * stdev
    lower = max(0, mean - 2 * stdev)
    print(f'  Baseline daily count: mean={mean:.1f}  stdev={stdev:.1f}  → expected [{lower:.1f}, {upper:.1f}]')
    print(f'  Today so far: {today_count} sign-ins')
    if today_count > upper:
        print(f'  🔴 ANOMALY: today\'s count exceeds +2σ band.')
    elif today_count < lower:
        print(f'  🟡 Lower-than-normal volume (could indicate an outage).')
    else:
        print(f'  ✅ Today is within normal range.')
else:
    print('  Not enough baseline days yet.')


## Step 3 — The hunting maturity model

Use this ladder to self-assess a hunting program:

| Level | Description | Example |
|-------|-------------|---------|
| 0 — **Initial** | No hunts. Reactive to alerts only. | "We look at Defender alerts when they fire." |
| 1 — **Ad-hoc hunts** | Manual, opportunistic, undocumented. | Someone greps logs after reading a news article. |
| 2 — **Hypothesis-driven** | Written hypotheses, repeatable queries, tied to MITRE ATT&CK. | "Hunt for T1059.001 PowerShell execution weekly." |
| 3 — **Data-driven / baseline** | Uses statistics on your own data: anomalies, UEBA, rare-in-environment. | Today's notebook. |
| 4 — **Automated** | Successful hunts become scheduled detections + playbooks. | Hunt 5 in notebook 1 → rule in notebook 2. |

The goal isn't to stay at level 4 forever — it's to constantly *promote* hunts up the ladder: ad-hoc → hypothesis → baseline → automated.


In [ ]:
# --- MITRE ATT&CK coverage view for this lab's hunts ---
hunts = [
    # (hunt_id,            tactic,            technique,  notebook)
    ('H-1 Failed signins', 'CredentialAccess','T1110',   '01'),
    ('H-2 Suspicious loc', 'InitialAccess',   'T1078',   '01'),
    ('H-3 Rare processes', 'Execution',       'T1059',   '01'),
    ('H-4 Bad-IP outbound','Exfiltration',    'T1041',   '01'),
    ('H-5 Brute success',  'CredentialAccess','T1110.001','01'),
    ('H2-1 Brute force',   'CredentialAccess','T1110',   '02'),
    ('H2-2 Lateral move',  'LateralMovement', 'T1021',   '02'),
    ('H2-3 Exfiltration',  'Exfiltration',    'T1041',   '02'),
    ('H2-4 TI watchlist',  '(cross-cutting)', 'IOC-match','02'),
    ('H3-1 Off-hours',     'InitialAccess',   'T1078',   '03'),
    ('H3-2 Volume anomaly','(behavioural/UEBA)', 'T1078',  '03'),
]

print(f'{"Hunt":<22} {"MITRE Tactic":<20} {"Technique":<12} Notebook')
print('-' * 65)
for h, tac, tech, nb in hunts:
    print(f'{h:<22} {tac:<20} {tech:<12} {nb}')

tactics = Counter(t for _, t, _, _ in hunts)
print('\nCoverage by MITRE ATT&CK tactic:')
for tac, c in tactics.most_common():
    bar = '█' * c
    print(f'  {tac:<22} {c} {bar}')

# The honest half of a coverage view is what is MISSING.
ENTERPRISE_TACTICS = [
    'Reconnaissance', 'ResourceDevelopment', 'InitialAccess', 'Execution',
    'Persistence', 'PrivilegeEscalation', 'DefenseEvasion', 'CredentialAccess',
    'Discovery', 'LateralMovement', 'Collection', 'CommandAndControl',
    'Exfiltration', 'Impact',
]
covered = {t for t in tactics if t in ENTERPRISE_TACTICS}
gaps = [t for t in ENTERPRISE_TACTICS if t not in covered]
print(f'\nCovered {len(covered)}/{len(ENTERPRISE_TACTICS)} enterprise tactics.')
print('❌ NO HUNT COVERS: ' + ', '.join(gaps))
print()
print('This is the point of an ATT&CK coverage view. A dashboard that only shows what you')
print('DO cover is a comfort blanket; the gap list is the backlog. Persistence and')
print('DefenseEvasion being empty means an attacker who survives a reboot or clears logs')
print('is invisible to every hunt in this lab.')


## You've completed all SC-200 labs!

### What you built and practiced

1. A **working mini-SIEM** with log ingestion, query engine, analytics rules, watchlists, and playbooks.
2. **Multi-stage attack investigation** across 4 data sources.
3. **Incident response workflows** — triage, investigate, contain, remediate, close.
4. **Threat hunting** at all four maturity levels — ad-hoc, hypothesis-driven, baseline/anomaly, and automated.
5. **MITRE ATT&CK mapping** for every hunt you wrote.

### Next steps

1. Take the [SC-200 practice assessment](https://learn.microsoft.com/en-us/credentials/certifications/security-operations-analyst/practice/assessment?assessment-type=practice&assessmentId=59&practice-assessment-type=certification)
2. Practice KQL at [detective.kusto.io](https://detective.kusto.io)
3. Explore real hunting queries: [Azure-Sentinel GitHub repo](https://github.com/Azure/Azure-Sentinel/tree/master/Hunting%20Queries)
4. Read the Microsoft Learn paths listed in the main README


---
## ✅ Self-check

1. Why is `bin(TimeGenerated % 1d, 1h)` wrong, and what are the two correct ways to get an
   hour-of-day bucket?
2. `series_decompose_anomalies()` returns what, and what must you do before you can `where`
   on the result?
3. In the anomaly flags array, what do `1`, `-1` and `0` mean?
4. Your off-hours rule pages the SOC every night. What is structurally wrong with it?
5. `bin(TimeGenerated, 1h)` vs `hourofday(TimeGenerated)` — when do you use each?
6. Your ATT&CK coverage chart shows five tactics covered and looks healthy. What is the
   more useful number?

In [ ]:
answers = """
1. KQL has no modulo operator for datetime, so `TimeGenerated % 1d` does not parse.
   Correct forms:
     hourofday(TimeGenerated)                             -> int 0..23
     bin(TimeGenerated - startofday(TimeGenerated), 1h)   -> timespan 00:00..23:00
   (dayofweek() and dayofmonth() are the sibling functions.)

2. It returns THREE columns as a tuple -- (anomaly flags, anomaly scores, baseline) --
   which you destructure:
     | extend (Flags, Scores, Baseline) = series_decompose_anomalies(Count, 2.5, -1, 'linefit')
   Because make-series produced ARRAY-valued columns, you must MV-EXPAND them back into
   rows (with `to typeof(...)`) before you can filter on individual points.

3.  1 = a positive anomaly / spike above the baseline
   -1 = a negative anomaly / dip below the baseline
    0 = within the expected band
   Dips matter too: a logging source going quiet is a classic defence-evasion signal.

4. It uses ONE weak signal in isolation. "Outside 09:00-17:00" describes half the day
   and most of your on-call engineers. Fixes: baseline per USER rather than a global
   window; require the hour to be unseen for THAT user; and combine it with other weak
   signals (new IP, new country, new app, privileged role) into a score, alerting only
   above a threshold. That is exactly what UEBA's investigation priority does.

5. bin(TimeGenerated, 1h) keeps the DATE and buckets the timeline -- use it for
   time-series (make-series, render timechart, per-window thresholds like "5 failures
   in 10 minutes").
   hourofday(TimeGenerated) DISCARDS the date and returns 0-23 -- use it to profile
   time-of-day behaviour across many days, which is what a baseline needs.

6. The number of tactics NOT covered, and which ones. Coverage charts flatter you: 5 of
   14 tactics covered means 9 blind spots. Persistence and Defense Evasion gaps in
   particular mean you would detect the break-in and miss the stay-in.
"""
print(answers)